# Notebook 02: Automate model governance with EventBridge and Model Cards

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> Tested with <strong>SageMaker Distribution Image 4.3.2</strong> and <strong>SageMaker Python SDK 3.20.0</strong>.
</div>

In this notebook, you automate completion and status updates for each registered model version's Model Card. Notebook 01's MLflow auto-sync created a draft card with training metrics and hyperparameters; an EventBridge-triggered Lambda adds the governance and evaluation details and keeps the status aligned for existing and pipeline-created versions.

|||
|---|---|
|00. Environment setup ||
|01. Experiment in a notebook, then scale with SageMaker AI jobs ||
|**02. Automate model governance with EventBridge and Model Cards** |**◄ YOU ARE HERE**|
|03. Operationalize with an ML pipeline and model registry ||
|04. Deploy to an endpoint and monitor ||
|05. Templatize staged deployment as a Project ||

## What you do here

1. Create an **EventBridge rule** on the `SageMaker Model Package State Change` event, filtered to your model package group(s).
2. Deploy a **Lambda function** ([`model_card_automation/lambda_function.py`](model_card_automation/lambda_function.py)) that completes the Model Card of the affected version and keeps the card status in step with the approval status.
3. **Backfill** the package you registered in notebook 01 and inspect the result.

**Outcome:** every registered version carries a completed Model Card whose status tracks the approval workflow, with no manual step.

## Architecture

![EventBridge invokes Lambda to enrich the Model Card on the same SageMaker Model Package version](img/model-card-automation-aws.svg)

<div class="alert alert-info">
<b>Which "model card"?</b> SageMaker has two flavors. The original <em>standalone</em> Model Card (<code>CreateModelCard</code>) is a separately named, separately versioned resource. Since the <a href="https://aws.amazon.com/blogs/machine-learning/improve-governance-of-models-with-amazon-sagemaker-unified-model-cards-and-model-registry/">unification with the Model Registry</a>, every <b>model package version carries its own model card</b> (the <code>ModelCard</code> block of <code>DescribeModelPackage</code>), with a strict one-to-one relationship, and that is what the Studio Model Registry UI shows under a version. This notebook works with that card. Creating a second, standalone card per version would duplicate the information and break the one-to-one link the registry gives you for free.
</div>

<div style="border: 4px solid #d32f2f; border-radius: 10px; padding: 20px; background-color: #ffebee; margin: 15px 0;">

### ⛔ IMPORTANT: Python 3.12 Required

All notebooks in this workshop **must** run on **Python 3.12**. The cell below will fail with an `AssertionError` if you are using a different Python version.

If you see an error, follow these steps:

1. Open a **Terminal** in JupyterLab (**File → New → Terminal**)
2. Run the setup script:
   ```bash
   cd ~/aim-408-end-to-end-modelops-on-sagemaker-ai
   ./setup-py312.sh
   ```
3. **Refresh your browser** to pick up the new kernel
4. In this notebook, go to **Kernel → Change Kernel → Python 3.12**
5. **Restart the kernel** and re-run from the [00-start-here.ipynb](./00-start-here.ipynb) cell to re-install the dependencies.

</div>

In [ ]:
import sys
assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info.major}.{sys.version_info.minor}. Please make sure you run on Python 3.12"

In [ ]:
%store -r 

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

## Setup

In [ ]:
import io
import json
import time
import zipfile
from pathlib import Path
from time import gmtime, strftime

import boto3
from botocore.exceptions import ClientError

# V3: Core imports
from sagemaker.core.helper.session_helper import Session, get_execution_role
from IPython.display import HTML, display

session = Session()
sm_role = get_execution_role()
region = session.boto_region_name
account_id = session.account_id()

sm_client = boto3.client("sagemaker")
iam_client = boto3.client("iam")
lambda_client = boto3.client("lambda")
events_client = boto3.client("events")

## Set constants

In [ ]:
project = "aim408-from-idea-to-prod"

# Reuse the MLflow registered model created in notebook 01 and restored via %store.
# Fall back to the same date-suffixed (-DDMMYY) name when running this standalone.
date_suffix = strftime('%d%m%y', gmtime())
if "registered_model_name" not in dir():
    registered_model_name = f"aim408-from-idea-to-prod-model-{date_suffix}"
    %store registered_model_name

# NOT the same string as the MLflow registered model: the auto-sync appends a short
# random suffix when it creates the SageMaker Model Package Group. Resolved for real
# against the registry in the backfill cell below; this is only a starting guess.
model_package_group_name = registered_model_name

# The EventBridge rule matches on this *prefix*, so it also covers the model package
# groups that other runs of the workshop create on other days. Notebook 04 registers
# nothing: it deploys the version notebook 03 registers into this same group.
model_package_group_prefix = f"{project}-model-"

model_card_lambda_name = f"{project}-model-card-automation"
model_card_lambda_role_name = f"{project}-model-card-automation-role"
model_card_rule_name = f"{project}-model-card-on-registration"

%store model_card_lambda_name
%store model_card_rule_name

print(f"MLflow registered model:  {registered_model_name}")
print(f"Rule matches groups:      {model_package_group_prefix}*")
print(f"Lambda function:          {model_card_lambda_name}")
print(f"EventBridge rule:         {model_card_rule_name}")

<div class="alert alert-warning">
<b>Permissions.</b> Besides the SageMaker permissions used so far, this notebook needs to deploy the automation: <code>iam:GetRole</code>, <code>iam:ListRolePolicies</code>, <code>iam:GetRolePolicy</code>, <code>iam:PassRole</code> (to <code>lambda.amazonaws.com</code>, for the Lambda execution role), <code>lambda:CreateFunction</code>, <code>lambda:UpdateFunctionCode</code>, <code>lambda:UpdateFunctionConfiguration</code>, <code>lambda:GetFunction</code>, <code>lambda:AddPermission</code>, <code>lambda:InvokeFunction</code>, <code>lambda:TagResource</code>, <code>events:PutRule</code>, <code>events:PutTargets</code>, <code>events:DescribeRule</code>, <code>events:TagResource</code>, and <code>logs:FilterLogEvents</code> to read the function's logs from here.
<br><br>
The workshop CloudFormation stack (<code>static/cfnSageMakerProjectsRoles.yaml</code>) grants all of that to the Studio execution role through the <code>ModelCardAutomationAccessPolicy</code> managed policy, scoped to the exact function and rule names set above — so if you rename them here, update <code>ModelCardLambdaFunctionName</code> / <code>ModelCardRuleName</code> in that template too. What the stack deliberately does <em>not</em> grant is <code>iam:CreateRole</code>, which is why the Lambda execution role is created by the same template rather than by the next cell. If you run this notebook outside the workshop account, set <code>LAMBDA_ROLE_ARN</code> below to a role your administrator created for you (trusts <code>lambda.amazonaws.com</code>, allows <code>sagemaker:DescribeModelPackage</code>, <code>sagemaker:UpdateModelPackage</code> and CloudWatch Logs).
</div>

## How the automation handles model versions

Before deploying anything, it is worth being precise about what the function does, because the Model Card rules are easy to get wrong:

| Recommendation (SageMaker Model Cards) | What the function does |
|---|---|
| Each model package version has exactly one model card | Writes the card **on the model package version** with `UpdateModelPackage(ModelCard=...)`. No standalone `CreateModelCard`, no card name to invent, nothing to keep in sync |
| A card version is immutable except for its status; any other change creates a new card version | Enriches the content **once** per package version and stamps it (`additional_information.custom_details.ModelCardAutomation`). Later events only touch `ModelCardStatus`, so re-deliveries do not pile up card versions |
| Card status has its own workflow: `Draft → PendingReview → Approved / Archived` | Maps the package's `ModelApprovalStatus` onto it: no status → `Draft`, `PendingManualApproval` → `PendingReview`, `Approved` → `Approved`, `Rejected` → `Archived` |
| Approved cards are frozen | Never edits the content of an `Approved` or `Archived` card. When it meets an already-approved package without an enriched card (your notebook 01 package), it writes the content as `PendingReview` first and approves in a second call |
| Keep training details from the source system | Keeps everything the MLflow auto-sync put on the card (training metrics, hyperparameters) and adds the governance sections around it |
| Populate evaluation results | Reads the `EvaluationMetrics` value that `pipeline_steps/register.py` writes into `CustomerMetadataProperties` (space-separated `name=value` pairs, because that field's charset rules out JSON) and renders it as `evaluation_details` |

And one operational rule of its own: **every invocation must be idempotent**. SageMaker may deliver several events per change, `register.py` updates the package right after the auto-sync created it, and the function's own `UpdateModelPackage` emits another event. The function always re-reads the package and exits without a write when the card already matches.

Have a look at the code; it is plain `boto3`, no extra dependencies, so it can be zipped and uploaded from this notebook.

In [ ]:
lambda_source_path = Path("model_card_automation/lambda_function.py")
lambda_source = lambda_source_path.read_text()

# Print the module docstring and the function signatures — open the file for the rest.
import ast
module = ast.parse(lambda_source)
print(ast.get_docstring(module))
print("\nFunctions:")
for node in module.body:
    if isinstance(node, ast.FunctionDef):
        print(f"  {node.name}({', '.join(a.arg for a in node.args.args)})")

## Look up the Lambda execution role

The function needs exactly two SageMaker permissions, scoped to the model package groups the rule matches, plus CloudWatch Logs. The SageMaker execution role you have used so far cannot be reused here: its trust policy allows `sagemaker.amazonaws.com`, not `lambda.amazonaws.com`, and it is far broader than what this function needs.

That role already exists. The workshop CloudFormation template `static/cfnSageMakerProjectsRoles.yaml` creates it as `ModelCardAutomationLambdaRole`, for the same reason it creates the two approval-gate persona roles you met in notebook 00: **the Studio execution role deliberately has no `iam:CreateRole`**. Least-privilege IAM belongs in the account's infrastructure-as-code, not in a notebook that any workshop participant can edit — so the notebook only looks the role up and passes it to Lambda.

The cell below resolves the role and prints the policy that is actually deployed on it, so you can check the blast radius of the function before you create it.

In [ ]:
# Override only if you run this notebook outside the workshop account and want to
# point the function at a role of your own. In the workshop, leave it at None.
LAMBDA_ROLE_ARN = None

if LAMBDA_ROLE_ARN is None:
    try:
        model_card_lambda_role_arn = iam_client.get_role(RoleName=model_card_lambda_role_name)["Role"]["Arn"]
        print(f"Using the pre-created role: {model_card_lambda_role_arn}")
    except iam_client.exceptions.NoSuchEntityException:
        raise RuntimeError(
            f"IAM role '{model_card_lambda_role_name}' does not exist.\n"
            "It is provisioned by the workshop CloudFormation stack as the resource\n"
            "ModelCardAutomationLambdaRole in static/cfnSageMakerProjectsRoles.yaml.\n"
            "Update that stack, or set LAMBDA_ROLE_ARN above to a role that trusts\n"
            "lambda.amazonaws.com and allows sagemaker:DescribeModelPackage,\n"
            "sagemaker:UpdateModelPackage and CloudWatch Logs."
        ) from None
else:
    model_card_lambda_role_arn = LAMBDA_ROLE_ARN
    print(f"Using provided role: {model_card_lambda_role_arn}")

# Print the least-privilege policy that is actually deployed on the role, so the
# blast radius of the function is visible before you create it.
role_name = model_card_lambda_role_arn.split("/")[-1]
try:
    for policy_name in iam_client.list_role_policies(RoleName=role_name)["PolicyNames"]:
        document = iam_client.get_role_policy(RoleName=role_name, PolicyName=policy_name)["PolicyDocument"]
        print(f"\nInline policy '{policy_name}':")
        for statement in document["Statement"]:
            actions = statement["Action"] if isinstance(statement["Action"], list) else [statement["Action"]]
            resources = statement["Resource"] if isinstance(statement["Resource"], list) else [statement["Resource"]]
            print(f"  {statement.get('Sid', '(no sid)'):<20}{statement['Effect']} {', '.join(actions)}")
            for resource in resources:
                print(f"  {'':<20}  on {resource}")
except ClientError as e:
    print(f"\n[WARN] Could not read the inline policies of {role_name} ({e.response['Error']['Code']}).")

%store model_card_lambda_role_arn

## Deploy the Lambda function

The deployment package is a zip with a single file. The governance text (purpose, intended uses, risk rating, stakeholders) is passed as environment variables, so you can change it per deployment without touching the code. Re-running the cell updates the code and configuration of an existing function.

In [ ]:
buffer = io.BytesIO()
with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(lambda_source_path, arcname="lambda_function.py")
zip_bytes = buffer.getvalue()

lambda_environment = {
    "Variables": {
        "MODEL_CARD_CREATOR": f"{project} workshop ({user_profile_name if 'user_profile_name' in dir() else 'sagemaker-user'})",
        "MODEL_CARD_PURPOSE": "Predict whether a bank customer will subscribe to a term deposit after a direct marketing call.",
        "MODEL_CARD_INTENDED_USES": "Rank customers for outbound marketing campaigns. Not intended for credit or eligibility decisions.",
        "MODEL_CARD_RISK_RATING": "Low",
        "MODEL_CARD_BUSINESS_PROBLEM": "Increase the conversion rate of term-deposit campaigns while reducing call volume.",
        "MODEL_CARD_STAKEHOLDERS": "Retail banking marketing team",
        "MODEL_CARD_LINE_OF_BUSINESS": "Retail banking",
    }
}

try:
    lambda_client.get_function(FunctionName=model_card_lambda_name)
    lambda_client.get_waiter("function_updated_v2").wait(FunctionName=model_card_lambda_name)
    lambda_client.update_function_code(FunctionName=model_card_lambda_name, ZipFile=zip_bytes)
    lambda_client.get_waiter("function_updated_v2").wait(FunctionName=model_card_lambda_name)
    lambda_client.update_function_configuration(
        FunctionName=model_card_lambda_name, Role=model_card_lambda_role_arn,
        Environment=lambda_environment, Timeout=60,
    )
    print(f"Updated function: {model_card_lambda_name}")
except lambda_client.exceptions.ResourceNotFoundException:
    for attempt in range(6):
        try:
            lambda_client.create_function(
                FunctionName=model_card_lambda_name,
                Runtime="python3.12",
                Role=model_card_lambda_role_arn,
                Handler="lambda_function.lambda_handler",
                Code={"ZipFile": zip_bytes},
                Description="Completes the Model Card of every model package version registered in the aim408-from-idea-to-prod groups",
                Timeout=60,
                MemorySize=256,
                Environment=lambda_environment,
                Tags={"project": project},
            )
            print(f"Created function: {model_card_lambda_name}")
            break
        except ClientError as e:
            # "The role defined for the function cannot be assumed by Lambda" = IAM propagation delay
            if e.response["Error"]["Code"] == "InvalidParameterValueException" and "assumed" in str(e) and attempt < 5:
                print("Role not yet assumable, retrying in 10s...")
                time.sleep(10)
            else:
                raise

lambda_client.get_waiter("function_active_v2").wait(FunctionName=model_card_lambda_name)
model_card_lambda_arn = lambda_client.get_function(FunctionName=model_card_lambda_name)["Configuration"]["FunctionArn"]
print(f"Function ARN: {model_card_lambda_arn}")

## Create the EventBridge rule

SageMaker publishes a [`SageMaker Model Package State Change`](https://docs.aws.amazon.com/sagemaker/latest/dg/automating-sagemaker-with-eventbridge.html#eventbridge-model-package) event to the default event bus whenever a model package version is created or updated. The event `detail` carries `ModelPackageGroupName`, `ModelPackageVersion`, `ModelPackageArn` and, once set, `ModelApprovalStatus`.

The rule filters on the model package group name with a `prefix` match, so it does not have to know the date suffix of today's group and it leaves other registries in the account alone. Then the function is registered as the rule's target and EventBridge is allowed to invoke it.

In [ ]:
event_pattern = {
    "source": ["aws.sagemaker"],
    "detail-type": ["SageMaker Model Package State Change"],
    "detail": {
        "ModelPackageGroupName": [{"prefix": model_package_group_prefix}],
    },
}

rule_arn = events_client.put_rule(
    Name=model_card_rule_name,
    EventPattern=json.dumps(event_pattern),
    State="ENABLED",
    Description="Complete the Model Card of every aim408-from-idea-to-prod model package version",
    Tags=[{"Key": "project", "Value": project}],
)["RuleArn"]
print(f"Rule: {rule_arn}")

events_client.put_targets(
    Rule=model_card_rule_name,
    Targets=[{"Id": "model-card-automation", "Arn": model_card_lambda_arn}],
)

try:
    lambda_client.add_permission(
        FunctionName=model_card_lambda_name,
        StatementId="allow-eventbridge-model-card-rule",
        Action="lambda:InvokeFunction",
        Principal="events.amazonaws.com",
        SourceArn=rule_arn,
    )
    print("EventBridge is allowed to invoke the function.")
except lambda_client.exceptions.ResourceConflictException:
    print("Invoke permission already present.")

print(json.dumps(event_pattern, indent=2))

## Backfill the version from notebook 01

The rule only fires on *future* state changes, so the version you registered in notebook 01 still has its bare draft card from the MLflow sync. Invoke the function directly with an event of the same shape EventBridge would send. This is also the quickest way to check the function works before the pipeline in notebook 03 depends on it.

Since that package is already `Approved`, expect the function to write the content as `PendingReview` and then approve the card in a second call (approved cards cannot be edited). The loop covers every version in the group, so it also documents any version registered before this point.

<div class="alert alert-warning">
<b>The group is not named after the MLflow model.</b> The auto-sync appends a short random suffix when it creates the Model Package Group, so the two names differ:

```
MLflow registered model   aim408-from-idea-to-prod-model-130926
Model Package Group       aim408-from-idea-to-prod-model-130926-9cd7c1
```

Notebook 01 never trips over this because it reads the `sagemaker.model_package_arn` tag that MLflow puts on the model version. Here there is no MLflow client, so the cell below searches the registry instead — the exact MLflow name first, then groups derived from it, then any `aim408-from-idea-to-prod-model-*` group. The prefix-matching EventBridge rule is unaffected, since the suffix is appended, not substituted.
</div>

In [ ]:
def model_package_event(summary):
    # Same shape as the EventBridge "SageMaker Model Package State Change" event
    return {
        "version": "0",
        "id": "manual-backfill",
        "detail-type": "SageMaker Model Package State Change",
        "source": "aws.sagemaker",
        "account": account_id,
        "region": region,
        "resources": [summary["ModelPackageArn"]],
        "detail": {
            "ModelPackageGroupName": model_package_group_name,
            "ModelPackageVersion": summary["ModelPackageVersion"],
            "ModelPackageArn": summary["ModelPackageArn"],
            "ModelApprovalStatus": summary.get("ModelApprovalStatus"),
        },
    }

def versions_in_group(group_name):
    """Versions of a model package group, oldest first.

    Returns [] for a group that does not exist: ListModelPackages does not raise on an
    unknown group name, so "group is empty" and "no such group" look identical here.
    """
    summaries = []
    paginator = sm_client.get_paginator("list_model_packages")
    for page in paginator.paginate(ModelPackageGroupName=group_name, SortBy="CreationTime", SortOrder="Ascending"):
        summaries.extend(page["ModelPackageSummaryList"])
    return summaries


def groups_containing(fragment):
    """Model package group names containing `fragment`, most recently created first."""
    # NameContains must match ^[a-zA-Z0-9](-*[a-zA-Z0-9]){0,62}$, so a trailing hyphen
    # (which model_package_group_prefix has) is rejected -- strip it before the call.
    names = []
    paginator = sm_client.get_paginator("list_model_package_groups")
    for page in paginator.paginate(
        NameContains=fragment.rstrip("-"), SortBy="CreationTime", SortOrder="Descending"
    ):
        names.extend(g["ModelPackageGroupName"] for g in page["ModelPackageGroupSummaryList"])
    return names


# Search narrowest first: the MLflow name as-is, then the groups the sync derived from
# it, then any group of this workshop (so an earlier day's run is still picked up).
search_order = [registered_model_name]
for name in groups_containing(registered_model_name) + [
    g for g in groups_containing(model_package_group_prefix) if g.startswith(model_package_group_prefix)
]:
    if name not in search_order:
        search_order.append(name)

packages = []
for candidate in search_order:
    packages = versions_in_group(candidate)
    if packages:
        model_package_group_name = candidate
        break

if not packages:
    raise RuntimeError(
        f"No model package version in any '{model_package_group_prefix}*' group in {region}.\n"
        f"Groups checked: {search_order}\n"
        "Run 01-mlflow-and-sagemaker-jobs.ipynb through the 'Register, and watch the "
        "auto-sync'\ncell in Part 2, then re-run this notebook from the top."
    )

if model_package_group_name != registered_model_name:
    print(f"MLflow registered model: {registered_model_name}")
print(f"Model Package Group:     {model_package_group_name} ({len(packages)} version(s))\n")

for summary in packages:
    test_event = model_package_event(summary)
    response = lambda_client.invoke(
        FunctionName=model_card_lambda_name,
        InvocationType="RequestResponse",
        Payload=json.dumps(test_event).encode(),
    )
    result = json.loads(response["Payload"].read())
    if response.get("FunctionError"):
        raise RuntimeError(f"Lambda failed for {summary['ModelPackageArn']}: {result}")
    print(f"v{summary['ModelPackageVersion']:<3} {summary.get('ModelApprovalStatus', '(no approval status)'):<24} -> {result['action']}")

# The most recent version is inspected in the next cells
model_package_arn = packages[-1]["ModelPackageArn"]
model_package_version = packages[-1]["ModelPackageVersion"]

### Inspect the completed model card

Read the card of the most recent version back from the model package. The training section is what the MLflow sync wrote; everything else came from the function.

In [ ]:
described = sm_client.describe_model_package(ModelPackageName=model_package_arn)
card = json.loads(described["ModelCard"]["ModelCardContent"])

print(f"Model package:   {model_package_group_name} v{model_package_version}")
print(f"Approval status: {described.get('ModelApprovalStatus', '(not set)')}")
print(f"Card status:     {described['ModelCard']['ModelCardStatus']}")

def section(title, obj):
    print(f"\n{title}")
    for k, v in obj.items():
        print(f"  {k:<36}{v if not isinstance(v, (dict, list)) else json.dumps(v)[:100]}")

section("Model overview", card.get("model_overview", {}))
section("Intended uses", card.get("intended_uses", {}))
section("Business details", card.get("business_details", {}))
for evaluation in card.get("evaluation_details", []):
    print(f"\nEvaluation: {evaluation['name']}")
    for group in evaluation.get("metric_groups", []):
        for m in group["metric_data"]:
            print(f"  {m['name']:<36}{m['value']}")
training = card.get("training_details", {}).get("training_job_details", {})
print("\nTraining metrics (from the MLflow sync)")
for m in training.get("training_metrics", []):
    print(f"  {m['name']:<36}{m['value']:.4f}")
section("Custom details", card.get("additional_information", {}).get("custom_details", {}))

In [ ]:
display(
    HTML('<b>See <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/registry/registered-models/{}/versions/version-{}">the model version</a> in the Studio Model Registry UI (Overview tab shows the model card, Audit shows its status)</b>'.format(
        domain_id, region, model_package_group_name, model_package_version
    ))
)

### Prove idempotency

Run the very same event again for the most recent version. SageMaker delivers duplicate events, and the function's own update produced one more a moment ago. None of them may create another card version.

In [ ]:
response = lambda_client.invoke(
    FunctionName=model_card_lambda_name,
    InvocationType="RequestResponse",
    Payload=json.dumps(test_event).encode(),
)
print(json.loads(response["Payload"].read())["action"])

### Watch the automation react to a real event (optional)

The rule has been live since you created it, so the function's own `UpdateModelPackage` calls from the backfill already produced real `Model Package State Change` events, which EventBridge routed back to the function. The CloudWatch log group shows those invocations ending in `no-op`. Reading the logs requires `logs:FilterLogEvents`; the cell only warns if you do not have it.

In [ ]:
logs_client = boto3.client("logs")
log_group = f"/aws/lambda/{model_card_lambda_name}"
try:
    time.sleep(15)  # give EventBridge + Lambda + CloudWatch a moment
    events = logs_client.filter_log_events(
        logGroupName=log_group,
        startTime=int((time.time() - 15 * 60) * 1000),
        filterPattern='"model-package/"',
    )["events"]
    for e in events[-10:]:
        print(e["message"].strip()[-160:])
    if not events:
        print("No log lines yet; re-run this cell in a minute.")
except Exception as e:
    print(f"[WARN] Could not read {log_group} ({e}). Open it in the CloudWatch console instead.")

## Every version, one card

Whenever the pipeline in notebook 03 registers a version, the same function runs: the version starts as `PendingManualApproval`, so its card lands in `PendingReview`; approving the package moves the card to `Approved`. The cell below lists all versions of the group with their approval and card status, so you can come back to it after notebook 03 and see the automation at work without any manual step.

In [ ]:
def model_card_overview(group_name):
    rows = []
    paginator = sm_client.get_paginator("list_model_packages")
    for page in paginator.paginate(ModelPackageGroupName=group_name, SortBy="CreationTime", SortOrder="Ascending"):
        for summary in page["ModelPackageSummaryList"]:
            d = sm_client.describe_model_package(ModelPackageName=summary["ModelPackageArn"])
            mc = d.get("ModelCard") or {}
            content = json.loads(mc["ModelCardContent"]) if mc.get("ModelCardContent") else {}
            custom = content.get("additional_information", {}).get("custom_details", {})
            rows.append({
                "version": d["ModelPackageVersion"],
                "approval": d.get("ModelApprovalStatus", "(not set)"),
                "card_status": mc.get("ModelCardStatus", "(no card)"),
                "automated": "yes" if custom.get("ModelCardAutomation") else "no",
                "risk": content.get("intended_uses", {}).get("risk_rating", ""),
                "created": d["CreationTime"].strftime("%Y-%m-%d %H:%M"),
            })
    return rows

print(f"{'ver':<5}{'approval':<24}{'card status':<15}{'automated':<11}{'risk':<8}created")
for r in model_card_overview(model_package_group_name):
    print(f"{r['version']:<5}{r['approval']:<24}{r['card_status']:<15}{r['automated']:<11}{r['risk']:<8}{r['created']}")

## Clean-up

**Leave the automation in place** — notebooks 03 and 04 register new versions and this is what documents them. The resources cost nothing while idle (a disabled-until-invoked Lambda function, an EventBridge rule, an IAM role).

When you are done with the whole workshop, flip the flag below to remove the function and the rule. The Lambda execution role is left alone: CloudFormation owns it, and deleting the workshop stack removes it.

In [ ]:
DELETE_AUTOMATION = False

if DELETE_AUTOMATION:
    try:
        events_client.remove_targets(Rule=model_card_rule_name, Ids=["model-card-automation"])
        events_client.delete_rule(Name=model_card_rule_name)
        print(f"Deleted rule {model_card_rule_name}")
    except ClientError as e:
        print(f"[WARN] rule: {e}")
    try:
        lambda_client.delete_function(FunctionName=model_card_lambda_name)
        print(f"Deleted function {model_card_lambda_name}")
    except ClientError as e:
        print(f"[WARN] function: {e}")
    # The Lambda execution role is not deleted here: it belongs to the workshop
    # CloudFormation stack and goes away when that stack is deleted.
    print(f"Left role {model_card_lambda_role_name} in place (owned by CloudFormation).")
else:
    print("Automation kept (DELETE_AUTOMATION=False).")

## Summary

In this notebook you:

- Learned how Model Cards relate to the Model Registry after their unification: **one card per model package version**, stored on the package itself, immutable except for its status
- Deployed a **Lambda function** that completes that card for any version registered in your model package groups and keeps the card status aligned with the approval workflow (`Draft → PendingReview → Approved / Archived`)
- Wired it to an **EventBridge rule** on `SageMaker Model Package State Change`, filtered by model package group prefix
- Backfilled the version from notebook 01 and verified that repeated events are **idempotent**, so duplicate deliveries and the function's own updates never create extra card versions

From here on, every registration — manual or from the pipeline — is documented without anyone remembering to do it.

## Continue with step 3

Open the step 3 [notebook](03-sagemaker-pipeline.ipynb). After the pipeline has registered a version, come back to the *Every version, one card* cell above, or use the check at the end of notebook 03.

## Further development ideas for your real-world projects

- **Wire the approval gate to deployment.** Notebooks 00, 01 and 03 already separate the personas: the data-scientist role can register and stage, only the governance-officer role can set `ModelApprovalStatus=Approved`. The automation here documents, the officer approves, and the resulting `Approved` event (`detail.ModelApprovalStatus`) is the natural trigger for a deployment pipeline
- Add a second target to the rule: an **SNS topic** or a **Step Functions** approval workflow that notifies reviewers when a card enters `PendingReview`
- Store the governance text (purpose, risk rating, stakeholders) per model package group as **group tags** and have the function read them, instead of function environment variables
- Add the **SHAP feature importances and fairness metrics** produced by the `explain` step to `evaluation_details`: have `register.py` pass them through `CustomerMetadataProperties` (or an S3 JSON referenced from the card) so the function can render them next to the test AUC
- Export approved cards to PDF with `CreateModelCardExportJob` for audit archives

## Additional resources

- [Amazon SageMaker Model Cards](https://docs.aws.amazon.com/sagemaker/latest/dg/model-cards.html)
- [Model card FAQs — versioning and Model Registry integration](https://docs.aws.amazon.com/sagemaker/latest/dg/model-cards-faqs.html)
- [Update the details of a model version (model package model card schema)](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry-details.html)
- [Improve governance of models with unified Model Cards and Model Registry](https://aws.amazon.com/blogs/machine-learning/improve-governance-of-models-with-amazon-sagemaker-unified-model-cards-and-model-registry/)
- [Automating Amazon SageMaker AI with Amazon EventBridge](https://docs.aws.amazon.com/sagemaker/latest/dg/automating-sagemaker-with-eventbridge.html)
- [EventBridge event patterns: prefix matching](https://docs.aws.amazon.com/eventbridge/latest/userguide/eb-event-patterns-content-based-filtering.html)

# Shutdown kernel

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>